<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/Akbar527/5_RecA_FDA_Drug_Repurposing_Workflow_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RecA FDA Drug Repurposing Workflow

Combined FDA drug prediction, PubChem retrieval, PaDEL fingerprinting, ExtraTrees prediction, docking template, and LIME explainability workflow.

## 05_FDA_drugs_prediction(1).py

In [7]:
# Install padelpy and lime if not already installed
!pip install padelpy
!pip install lime

In [12]:
import argparse
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import padelpy
import requests

from lime.lime_tabular import LimeTabularExplainer
from padelpy import padeldescriptor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.pipeline import Pipeline


SCRIPT_DIR = Path.cwd()

FDA_INPUT_FILE = SCRIPT_DIR / "fda_drug_seed_list.csv"
TRAINING_DATA_FILE = SCRIPT_DIR / "03_recA_low_variance_filtered.csv"
F_SCORE_FILE = SCRIPT_DIR / "03_recA_fscore_ranking.csv"

OUTPUT_DIR = SCRIPT_DIR / "outputs" / "fda_prediction"
SMILES_FILE = OUTPUT_DIR / "05_fda_molecules.smi"
PUBCHEM_FILE = OUTPUT_DIR / "05_pubchem_fda_drugs.csv"
FINGERPRINT_FILE = OUTPUT_DIR / "05_fda_combined_fingerprints.csv"
PREDICTION_FILE = OUTPUT_DIR / "05_fda_recA_predictions.csv"
DOCKING_TEMPLATE_FILE = OUTPUT_DIR / "05_docking_results_template.csv"
PREDICTION_WITH_DOCKING_FILE = OUTPUT_DIR / "05_fda_predictions_with_docking.csv"
LIME_DIR = OUTPUT_DIR / "lime"

TOP_K_FEATURES = 100
TOP_N_LIME = 5
RANDOM_STATE = 42

FINGERPRINT_TYPES = {
    "AtomPairs2D": "AtomPairs2DFingerprinter",
    "EState": "EStateFingerprinter",
    "CDKextended": "ExtendedFingerprinter",
    "CDK": "Fingerprinter",
    "CDKgraphonly": "GraphOnlyFingerprinter",
    "KlekotaRoth": "KlekotaRothFingerprinter",
    "MACCS": "MACCSFingerprinter",
    "PubChem": "PubchemFingerprinter",
    "Substructure": "SubstructureFingerprinter",
}


def check_file_exists(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"\nFile not found:\n{path}\n\n{message}")


def load_seed_drugs() -> list[str]:
    check_file_exists(
        FDA_INPUT_FILE,
        "Please place fda_drug_seed_list.csv in the project folder.",
    )

    df = pd.read_csv(FDA_INPUT_FILE)

    if "compound_name" not in df.columns:
        raise ValueError("FDA seed file must contain column: compound_name")

    return df["compound_name"].dropna().astype(str).str.strip().tolist()


def fetch_pubchem_record(compound_name: str) -> dict | None:
    properties = [
        "Title",
        "CanonicalSMILES",
        "ConnectivitySMILES",
        "IsomericSMILES",
        "MolecularFormula",
        "MolecularWeight",
        "XLogP",
        "TPSA",
        "HBondDonorCount",
        "HBondAcceptorCount",
        "RotatableBondCount",
    ]

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{requests.utils.quote(compound_name)}"
        f"/property/{','.join(properties)}/JSON"
    )

    try:
        response = requests.get(url, timeout=60)
    except requests.RequestException as error:
        print(f"Failed to fetch {compound_name}: {error}")
        return None

    if response.status_code != 200:
        print(f"PubChem not found: {compound_name} | status={response.status_code}")
        return None

    try:
        data = response.json().get("PropertyTable", {}).get("Properties", [])
    except ValueError:
        print(f"Invalid PubChem JSON response for: {compound_name}")
        return None

    if not data:
        print(f"No PubChem data found for: {compound_name}")
        return None

    record = data[0]

    smiles = (
        record.get("CanonicalSMILES")
        or record.get("ConnectivitySMILES")
        or record.get("IsomericSMILES")
    )

    if not smiles:
        print(f"No SMILES found for: {compound_name}")
        return None

    return {
        "compound_name": compound_name,
        "pubchem_title": record.get("Title", compound_name),
        "cid": record.get("CID"),
        "canonical_smiles": smiles,
        "molecular_formula": record.get("MolecularFormula"),
        "molecular_weight": record.get("MolecularWeight"),
        "xlogp": record.get("XLogP"),
        "tpsa": record.get("TPSA"),
        "hbd": record.get("HBondDonorCount"),
        "hba": record.get("HBondAcceptorCount"),
        "rotatable_bonds": record.get("RotatableBondCount"),
    }


def fetch_pubchem_fda_drugs() -> pd.DataFrame:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rows = []

    for compound_name in load_seed_drugs():
        print(f"Fetching PubChem record: {compound_name}")

        record = fetch_pubchem_record(compound_name)

        if record and record.get("canonical_smiles"):
            rows.append(record)

    if not rows:
        raise RuntimeError("No valid PubChem records were retrieved.")

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["cid"])
        .reset_index(drop=True)
    )

    df.to_csv(PUBCHEM_FILE, index=False)

    return df


def write_smiles_file(df: pd.DataFrame) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    required = ["canonical_smiles", "cid"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(f"Missing PubChem columns: {missing}")

    lines = [
        f"{smiles}\tCID{cid}"
        for smiles, cid in df[["canonical_smiles", "cid"]].itertuples(index=False)
    ]

    SMILES_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")


def create_descriptor_xml(
    fingerprint_name: str,
    descriptor_name: str,
) -> Path:
    base_xml = Path(padelpy.__file__).parent / "PaDEL-Descriptor" / "descriptors.xml"

    if not base_xml.exists():
        raise FileNotFoundError(f"PaDEL descriptors.xml not found:\n{base_xml}")

    output_xml = OUTPUT_DIR / f"05_{fingerprint_name}.xml"

    tree = ET.parse(base_xml)
    root = tree.getroot()

    found = False

    for descriptor in root.iter("Descriptor"):
        is_target = descriptor.attrib.get("name") == descriptor_name
        descriptor.set("value", "true" if is_target else "false")

        if is_target:
            found = True

    if not found:
        raise ValueError(f"Descriptor {descriptor_name} not found in {base_xml}")

    tree.write(output_xml, encoding="utf-8", xml_declaration=False)

    return output_xml


def calculate_fingerprints() -> pd.DataFrame:
    all_descriptors: pd.DataFrame | None = None

    for fingerprint_name, descriptor_name in FINGERPRINT_TYPES.items():
        print(f"Calculating {fingerprint_name} fingerprints...")

        descriptor_xml = create_descriptor_xml(
            fingerprint_name,
            descriptor_name,
        )

        output_file = OUTPUT_DIR / f"05_{fingerprint_name}.csv"

        padeldescriptor(
            mol_dir=str(SMILES_FILE.resolve()),
            d_file=str(output_file.resolve()),
            descriptortypes=str(descriptor_xml.resolve()),
            detectaromaticity=True,
            standardizenitro=True,
            standardizetautomers=True,
            threads=2,
            removesalt=True,
            log=True,
            fingerprints=True,
            sp_timeout=600,
        )

        if not output_file.exists():
            raise RuntimeError(f"PaDEL failed to generate: {output_file}")

        df = pd.read_csv(output_file)

        if "Name" not in df.columns:
            raise ValueError(f"'Name' column not found in {output_file}")

        if all_descriptors is None:
            all_descriptors = df
        else:
            all_descriptors = all_descriptors.merge(df, on="Name", how="inner")

    if all_descriptors is None:
        raise RuntimeError("No fingerprints were generated for FDA compounds.")

    all_descriptors.to_csv(FINGERPRINT_FILE, index=False)

    return all_descriptors


def load_training_data(
    top_k: int = TOP_K_FEATURES,
) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    check_file_exists(
        TRAINING_DATA_FILE,
        "Please run 03_Feature_selection_low_variance_Fscore.py first.",
    )

    check_file_exists(
        F_SCORE_FILE,
        "Please run 03_Feature_selection_low_variance_Fscore.py first.",
    )

    training = pd.read_csv(TRAINING_DATA_FILE)
    ranking = pd.read_csv(F_SCORE_FILE)

    if "feature" not in ranking.columns:
        raise ValueError("F-score ranking file must contain column: feature")

    if "class" not in training.columns:
        raise ValueError("Training data must contain column: class")

    selected_features = [
        feature
        for feature in ranking["feature"].head(top_k).tolist()
        if feature in training.columns
    ]

    if not selected_features:
        raise ValueError("No selected features were found in training data.")

    x_train = (
        training[selected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y_train = training["class"].astype(int)

    return x_train, y_train, selected_features


def align_prediction_features(
    fingerprint_df: pd.DataFrame,
    selected_features: list[str],
) -> pd.DataFrame:
    aligned = fingerprint_df.copy()

    if "Name" not in aligned.columns:
        raise ValueError("Fingerprint dataframe must contain column: Name")

    for feature in selected_features:
        if feature not in aligned.columns:
            aligned[feature] = 0

    aligned_features = (
        aligned[["Name", *selected_features]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return aligned_features


def train_prediction_model(
    x_train: pd.DataFrame,
    y_train: pd.Series,
) -> Pipeline:
    model = Pipeline(
        [
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=min(40, x_train.shape[1]))),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=180,
                    min_samples_leaf=2,
                    max_features=0.4,
                    bootstrap=False,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )

    model.fit(x_train, y_train)

    return model


def predict_fda_drugs(
    pubchem_df: pd.DataFrame,
    aligned_fingerprints: pd.DataFrame,
    model: Pipeline,
    selected_features: list[str],
) -> pd.DataFrame:
    x_pred = aligned_fingerprints[selected_features].apply(
        pd.to_numeric,
        errors="coerce",
    ).fillna(0)

    probabilities = model.predict_proba(x_pred)[:, 1]

    results = aligned_fingerprints[["Name"]].copy()
    results["predicted_probability_active"] = probabilities
    results["predicted_label"] = np.where(
        probabilities >= 0.5,
        "active_like",
        "inactive_like",
    )

    results["cid"] = (
        results["Name"]
        .str.replace("CID", "", regex=False)
        .astype(int)
    )

    merged = pubchem_df.merge(
        results,
        on="cid",
        how="inner",
    )

    merged = (
        merged
        .sort_values("predicted_probability_active", ascending=False)
        .reset_index(drop=True)
    )

    merged.to_csv(PREDICTION_FILE, index=False)

    return merged


def prepare_docking_template(predictions: pd.DataFrame) -> pd.DataFrame:
    top_predictions = predictions.head(10).copy()

    docking_template = top_predictions[
        [
            "cid",
            "compound_name",
            "pubchem_title",
            "canonical_smiles",
            "predicted_probability_active",
            "predicted_label",
        ]
    ].copy()

    docking_template["pdb_id"] = "1MO3"
    docking_template["binding_affinity_kcal_mol"] = ""
    docking_template["docking_rank"] = ""
    docking_template["binding_site_notes"] = ""

    docking_template.to_csv(DOCKING_TEMPLATE_FILE, index=False)

    return docking_template


def merge_with_docking(predictions: pd.DataFrame) -> pd.DataFrame:
    if DOCKING_TEMPLATE_FILE.exists():
        docking = pd.read_csv(DOCKING_TEMPLATE_FILE)
    else:
        docking = prepare_docking_template(predictions)

    merge_cols = [
        "cid",
        "compound_name",
        "pubchem_title",
        "predicted_probability_active",
        "predicted_label",
    ]

    merged = predictions.merge(
        docking,
        on=merge_cols,
        how="left",
        suffixes=("", "_docking"),
    )

    merged.to_csv(PREDICTION_WITH_DOCKING_FILE, index=False)

    return merged


def generate_lime_explanations(
    model: Pipeline,
    x_train: pd.DataFrame,
    predictions: pd.DataFrame,
    aligned_fingerprints: pd.DataFrame,
    selected_features: list[str],
    top_n: int = TOP_N_LIME,
) -> list[Path]:
    LIME_DIR.mkdir(parents=True, exist_ok=True)

    explainer = LimeTabularExplainer(
        training_data=x_train[selected_features].values,
        feature_names=selected_features,
        class_names=["inactive_like", "active_like"],
        mode="classification",
        discretize_continuous=False,
    )

    output_files: list[Path] = []
    top_predictions = predictions.head(top_n)
    feature_frame = aligned_fingerprints.set_index("Name")

    for _, row in top_predictions.iterrows():
        name = f"CID{row['cid']}"

        if name not in feature_frame.index:
            continue

        instance = feature_frame.loc[name, selected_features].values

        explanation = explainer.explain_instance(
            instance,
            model.predict_proba,
            num_features=10,
        )

        html_file = LIME_DIR / f"{name}_lime.html"
        png_file = LIME_DIR / f"{name}_lime.png"

        explanation.save_to_file(str(html_file))

        fig = explanation.as_pyplot_figure()
        fig.tight_layout()
        fig.savefig(png_file, dpi=300, bbox_inches="tight")
        plt.close(fig)

        output_files.extend([html_file, png_file])

    return output_files


def run_workflow(top_k: int = TOP_K_FEATURES) -> dict[str, object]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    pubchem_df = fetch_pubchem_fda_drugs()
    write_smiles_file(pubchem_df)

    fingerprints = calculate_fingerprints()

    x_train, y_train, selected_features = load_training_data(
        top_k=top_k,
    )

    aligned_fingerprints = align_prediction_features(
        fingerprints,
        selected_features,
    )

    model = train_prediction_model(
        x_train,
        y_train,
    )

    predictions = predict_fda_drugs(
        pubchem_df,
        aligned_fingerprints,
        model,
        selected_features,
    )

    docking = merge_with_docking(predictions)

    lime_files = generate_lime_explanations(
        model,
        x_train,
        predictions,
        aligned_fingerprints,
        selected_features,
    )

    return {
        "pubchem_df": pubchem_df,
        "fingerprints": fingerprints,
        "selected_features": selected_features,
        "predictions": predictions,
        "docking": docking,
        "lime_files": lime_files,
    }


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Predict RecA inhibitor likelihood for a starter panel of "
            "FDA-approved drugs fetched from PubChem."
        )
    )

    parser.add_argument(
        "--top-k",
        type=int,
        default=TOP_K_FEATURES,
        help="Number of selected fingerprint features.",
    )

    # In interactive environments like Colab, argparse can receive unrecognized arguments
    # from the kernel. parse_known_args() allows parsing only the arguments defined
    # in the script and ignores the others.
    args, unknown = parser.parse_known_args()

    results = run_workflow(top_k=args.top_k)
    predictions = results["predictions"]

    print(f"Retrieved PubChem drugs: {len(results['pubchem_df'])}")
    print(f"Fingerprint shape: {results['fingerprints'].shape}")
    print(f"Prediction count: {len(predictions)}")

    print("\nTop predicted compounds:")
    print(
        predictions[
            [
                "compound_name",
                "cid",
                "predicted_probability_active",
                "predicted_label",
            ]
        ].head(10).to_string(index=False)
    )

    print(f"\nLIME files generated: {len(results['lime_files'])}")
    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Fetching PubChem record: prothionamide
Fetching PubChem record: clofazimine
Fetching PubChem record: rifapentine
Fetching PubChem record: pyrazinamide
Fetching PubChem record: atorvastatin
Fetching PubChem record: isoniazid
Fetching PubChem record: pretomanid
Fetching PubChem record: bedaquiline
Fetching PubChem record: doxycycline
Fetching PubChem record: isoniazid sodium methanesulfonate
Fetching PubChem record: ibuprofen
Fetching PubChem record: ethionamide
Fetching PubChem record: rifampicin
Fetching PubChem record: omeprazole
Fetching PubChem record: meropenem
Fetching PubChem record: moxifloxacin
Fetching PubChem record: fluconazole
Fetching PubChem record: azithromycin
Fetching PubChem record: ciprofloxacin
Fetching PubChem record: levofloxacin
Fetching PubChem record: ethambutol
Fetching PubChem record: metformin
Fetching PubChem record: delamanid
Fetching PubChem record: amoxicillin
Fetching PubChem record: rifabutin
Fetching PubChem record: linezolid
Fetching PubChem record: 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(


Retrieved PubChem drugs: 31
Fingerprint shape: (31, 10146)
Prediction count: 31

Top predicted compounds:
                    compound_name       cid  predicted_probability_active predicted_label
                    prothionamide    666418                      0.638495     active_like
                      doxycycline  54671203                      0.627288     active_like
                       rifampicin 135398735                      0.618995     active_like
                     pyrazinamide      1046                      0.519179     active_like
                      rifapentine 135403821                      0.506389     active_like
                       omeprazole      4594                      0.490939   inactive_like
                      clofazimine      2794                      0.442302   inactive_like
isoniazid sodium methanesulfonate  23691029                      0.431678   inactive_like
                     levofloxacin    149096                      0.406019   inactive

## 05_FDA_drugs_prediction_fixed(1).py

In [15]:
import argparse
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import padelpy
import requests

from lime.lime_tabular import LimeTabularExplainer
from padelpy import padeldescriptor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.pipeline import Pipeline


SCRIPT_DIR = Path.cwd()

FDA_INPUT_FILE = SCRIPT_DIR / "fda_drug_seed_list.csv"
TRAINING_DATA_FILE = SCRIPT_DIR / "03_recA_low_variance_filtered.csv"
F_SCORE_FILE = SCRIPT_DIR / "03_recA_fscore_ranking.csv"

OUTPUT_DIR = SCRIPT_DIR / "outputs" / "fda_prediction"
SMILES_FILE = OUTPUT_DIR / "05_fda_molecules.smi"
PUBCHEM_FILE = OUTPUT_DIR / "05_pubchem_fda_drugs.csv"
FINGERPRINT_FILE = OUTPUT_DIR / "05_fda_combined_fingerprints.csv"
PREDICTION_FILE = OUTPUT_DIR / "05_fda_recA_predictions.csv"
DOCKING_TEMPLATE_FILE = OUTPUT_DIR / "05_docking_results_template.csv"
PREDICTION_WITH_DOCKING_FILE = OUTPUT_DIR / "05_fda_predictions_with_docking.csv"
LIME_DIR = OUTPUT_DIR / "lime"

TOP_K_FEATURES = 100
TOP_N_LIME = 5
RANDOM_STATE = 42

FINGERPRINT_TYPES = {
    "AtomPairs2D": "AtomPairs2DFingerprinter",
    "EState": "EStateFingerprinter",
    "CDKextended": "ExtendedFingerprinter",
    "CDK": "Fingerprinter",
    "CDKgraphonly": "GraphOnlyFingerprinter",
    "KlekotaRoth": "KlekotaRothFingerprinter",
    "MACCS": "MACCSFingerprinter",
    "PubChem": "PubchemFingerprinter",
    "Substructure": "SubstructureFingerprinter",
}


def check_file_exists(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"\nFile not found:\n{path}\n\n{message}")


def load_seed_drugs() -> list[str]:
    check_file_exists(
        FDA_INPUT_FILE,
        "Please place fda_drug_seed_list.csv in the project folder.",
    )

    df = pd.read_csv(FDA_INPUT_FILE)

    if "compound_name" not in df.columns:
        raise ValueError("FDA seed file must contain column: compound_name")

    return df["compound_name"].dropna().astype(str).str.strip().tolist()


def fetch_pubchem_record(compound_name: str) -> dict | None:
    properties = [
        "Title",
        "CanonicalSMILES",
        "ConnectivitySMILES",
        "IsomericSMILES",
        "MolecularFormula",
        "MolecularWeight",
        "XLogP",
        "TPSA",
        "HBondDonorCount",
        "HBondAcceptorCount",
        "RotatableBondCount",
    ]

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{requests.utils.quote(compound_name)}"
        f"/property/{','.join(properties)}/JSON"
    )

    try:
        response = requests.get(url, timeout=60)
    except requests.RequestException as error:
        print(f"Failed to fetch {compound_name}: {error}")
        return None

    if response.status_code != 200:
        print(f"PubChem not found: {compound_name} | status={response.status_code}")
        return None

    try:
        data = response.json().get("PropertyTable", {}).get("Properties", [])
    except ValueError:
        print(f"Invalid PubChem JSON response for: {compound_name}")
        return None

    if not data:
        print(f"No PubChem data found for: {compound_name}")
        return None

    record = data[0]

    smiles = (
        record.get("CanonicalSMILES")
        or record.get("ConnectivitySMILES")
        or record.get("IsomericSMILES")
    )

    if not smiles:
        print(f"No SMILES found for: {compound_name}")
        return None

    return {
        "compound_name": compound_name,
        "pubchem_title": record.get("Title", compound_name),
        "cid": record.get("CID"),
        "canonical_smiles": smiles,
        "molecular_formula": record.get("MolecularFormula"),
        "molecular_weight": record.get("MolecularWeight"),
        "xlogp": record.get("XLogP"),
        "tpsa": record.get("TPSA"),
        "hbd": record.get("HBondDonorCount"),
        "hba": record.get("HBondAcceptorCount"),
        "rotatable_bonds": record.get("RotatableBondCount"),
    }


def fetch_pubchem_fda_drugs() -> pd.DataFrame:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rows = []

    for compound_name in load_seed_drugs():
        print(f"Fetching PubChem record: {compound_name}")

        record = fetch_pubchem_record(compound_name)

        if record and record.get("canonical_smiles"):
            rows.append(record)

    if not rows:
        raise RuntimeError("No valid PubChem records were retrieved.")

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["cid"])
        .reset_index(drop=True)
    )

    df.to_csv(PUBCHEM_FILE, index=False)

    return df


def write_smiles_file(df: pd.DataFrame) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    required = ["canonical_smiles", "cid"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(f"Missing PubChem columns: {missing}")

    lines = [
        f"{smiles}\tCID{cid}"
        for smiles, cid in df[["canonical_smiles", "cid"]].itertuples(index=False)
    ]

    SMILES_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")


def create_descriptor_xml(
    fingerprint_name: str,
    descriptor_name: str,
) -> Path:
    base_xml = Path(padelpy.__file__).parent / "PaDEL-Descriptor" / "descriptors.xml"

    if not base_xml.exists():
        raise FileNotFoundError(f"PaDEL descriptors.xml not found:\n{base_xml}")

    output_xml = OUTPUT_DIR / f"05_{fingerprint_name}.xml"

    tree = ET.parse(base_xml)
    root = tree.getroot()

    found = False

    for descriptor in root.iter("Descriptor"):
        is_target = descriptor.attrib.get("name") == descriptor_name
        descriptor.set("value", "true" if is_target else "false")

        if is_target:
            found = True

    if not found:
        raise ValueError(f"Descriptor {descriptor_name} not found in {base_xml}")

    tree.write(output_xml, encoding="utf-8", xml_declaration=False)

    return output_xml


def calculate_fingerprints() -> pd.DataFrame:
    all_descriptors: pd.DataFrame | None = None

    for fingerprint_name, descriptor_name in FINGERPRINT_TYPES.items():
        print(f"Calculating {fingerprint_name} fingerprints...")

        descriptor_xml = create_descriptor_xml(
            fingerprint_name,
            descriptor_name,
        )

        output_file = OUTPUT_DIR / f"05_{fingerprint_name}.csv"

        padeldescriptor(
            mol_dir=str(SMILES_FILE.resolve()),
            d_file=str(output_file.resolve()),
            descriptortypes=str(descriptor_xml.resolve()),
            detectaromaticity=True,
            standardizenitro=True,
            standardizetautomers=True,
            threads=2,
            removesalt=True,
            log=True,
            fingerprints=True,
            sp_timeout=600,
        )

        if not output_file.exists():
            raise RuntimeError(f"PaDEL failed to generate: {output_file}")

        df = pd.read_csv(output_file)

        if "Name" not in df.columns:
            raise ValueError(f"'Name' column not found in {output_file}")

        if all_descriptors is None:
            all_descriptors = df
        else:
            all_descriptors = all_descriptors.merge(df, on="Name", how="inner")

    if all_descriptors is None:
        raise RuntimeError("No fingerprints were generated for FDA compounds.")

    all_descriptors.to_csv(FINGERPRINT_FILE, index=False)

    return all_descriptors


def load_training_data(
    top_k: int = TOP_K_FEATURES,
) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    check_file_exists(
        TRAINING_DATA_FILE,
        "Please run 03_Feature_selection_low_variance_Fscore.py first.",
    )

    check_file_exists(
        F_SCORE_FILE,
        "Please run 03_Feature_selection_low_variance_Fscore.py first.",
    )

    training = pd.read_csv(TRAINING_DATA_FILE)
    ranking = pd.read_csv(F_SCORE_FILE)

    if "feature" not in ranking.columns:
        raise ValueError("F-score ranking file must contain column: feature")

    if "class" not in training.columns:
        raise ValueError("Training data must contain column: class")

    selected_features = [
        feature
        for feature in ranking["feature"].head(top_k).tolist()
        if feature in training.columns
    ]

    if not selected_features:
        raise ValueError("No selected features were found in training data.")

    x_train = (
        training[selected_features]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y_train = training["class"].astype(int)

    return x_train, y_train, selected_features


def align_prediction_features(
    fingerprint_df: pd.DataFrame,
    selected_features: list[str],
) -> pd.DataFrame:
    aligned = fingerprint_df.copy()

    if "Name" not in aligned.columns:
        raise ValueError("Fingerprint dataframe must contain column: Name")

    for feature in selected_features:
        if feature not in aligned.columns:
            aligned[feature] = 0

    aligned_features = (
        aligned[["Name", *selected_features]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return aligned_features


def train_prediction_model(
    x_train: pd.DataFrame,
    y_train: pd.Series,
) -> Pipeline:
    model = Pipeline(
        [
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=min(40, x_train.shape[1]))),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=180,
                    min_samples_leaf=2,
                    max_features=0.4,
                    bootstrap=False,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )

    model.fit(x_train, y_train)

    return model


def predict_fda_drugs(
    pubchem_df: pd.DataFrame,
    aligned_fingerprints: pd.DataFrame,
    model: Pipeline,
    selected_features: list[str],
) -> pd.DataFrame:
    x_pred = aligned_fingerprints[selected_features].apply(
        pd.to_numeric,
        errors="coerce",
    ).fillna(0)

    probabilities = model.predict_proba(x_pred)[:, 1]

    results = aligned_fingerprints[["Name"]].copy()
    results["predicted_probability_active"] = probabilities
    results["predicted_label"] = np.where(
        probabilities >= 0.5,
        "active_like",
        "inactive_like",
    )

    results["cid"] = (
        results["Name"]
        .str.replace("CID", "", regex=False)
        .astype(int)
    )

    merged = pubchem_df.merge(
        results,
        on="cid",
        how="inner",
    )

    merged = (
        merged
        .sort_values("predicted_probability_active", ascending=False)
        .reset_index(drop=True)
    )

    merged.to_csv(PREDICTION_FILE, index=False)

    return merged


def prepare_docking_template(predictions: pd.DataFrame) -> pd.DataFrame:
    top_predictions = predictions.head(10).copy()

    docking_template = top_predictions[
        [
            "cid",
            "compound_name",
            "pubchem_title",
            "canonical_smiles",
            "predicted_probability_active",
            "predicted_label",
        ]
    ].copy()

    docking_template["pdb_id"] = "1MO3"
    docking_template["binding_affinity_kcal_mol"] = ""
    docking_template["docking_rank"] = ""
    docking_template["binding_site_notes"] = ""

    docking_template.to_csv(DOCKING_TEMPLATE_FILE, index=False)

    return docking_template


def merge_with_docking(predictions: pd.DataFrame) -> pd.DataFrame:
    if DOCKING_TEMPLATE_FILE.exists():
        docking = pd.read_csv(DOCKING_TEMPLATE_FILE)
    else:
        docking = prepare_docking_template(predictions)

    merge_cols = [
        "cid",
        "compound_name",
        "pubchem_title",
        "predicted_probability_active",
        "predicted_label",
    ]

    merged = predictions.merge(
        docking,
        on=merge_cols,
        how="left",
        suffixes=("", "_docking"),
    )

    merged.to_csv(PREDICTION_WITH_DOCKING_FILE, index=False)

    return merged


def generate_lime_explanations(
    model: Pipeline,
    x_train: pd.DataFrame,
    predictions: pd.DataFrame,
    aligned_fingerprints: pd.DataFrame,
    selected_features: list[str],
    top_n: int = TOP_N_LIME,
) -> list[Path]:
    LIME_DIR.mkdir(parents=True, exist_ok=True)

    explainer = LimeTabularExplainer(
        training_data=x_train[selected_features].values,
        feature_names=selected_features,
        class_names=["inactive_like", "active_like"],
        mode="classification",
        discretize_continuous=False,
    )

    output_files: list[Path] = []
    top_predictions = predictions.head(top_n)
    feature_frame = aligned_fingerprints.set_index("Name")

    for _, row in top_predictions.iterrows():
        name = f"CID{row['cid']}"

        if name not in feature_frame.index:
            continue

        instance = feature_frame.loc[name, selected_features].values

        explanation = explainer.explain_instance(
            instance,
            model.predict_proba,
            num_features=10,
        )

        html_file = LIME_DIR / f"{name}_lime.html"
        png_file = LIME_DIR / f"{name}_lime.png"

        explanation.save_to_file(str(html_file))

        fig = explanation.as_pyplot_figure()
        fig.tight_layout()
        fig.savefig(png_file, dpi=300, bbox_inches="tight")
        plt.close(fig)

        output_files.extend([html_file, png_file])

    return output_files


def run_workflow(top_k: int = TOP_K_FEATURES) -> dict[str, object]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    pubchem_df = fetch_pubchem_fda_drugs()
    write_smiles_file(pubchem_df)

    fingerprints = calculate_fingerprints()

    x_train, y_train, selected_features = load_training_data(
        top_k=top_k,
    )

    aligned_fingerprints = align_prediction_features(
        fingerprints,
        selected_features,
    )

    model = train_prediction_model(
        x_train,
        y_train,
    )

    predictions = predict_fda_drugs(
        pubchem_df,
        aligned_fingerprints,
        model,
        selected_features,
    )

    docking = merge_with_docking(predictions)

    lime_files = generate_lime_explanations(
        model,
        x_train,
        predictions,
        aligned_fingerprints,
        selected_features,
    )

    return {
        "pubchem_df": pubchem_df,
        "fingerprints": fingerprints,
        "selected_features": selected_features,
        "predictions": predictions,
        "docking": docking,
        "lime_files": lime_files,
    }


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Predict RecA inhibitor likelihood for a starter panel of "
            "FDA-approved drugs fetched from PubChem."
        )
    )

    parser.add_argument(
        "--top-k",
        type=int,
        default=TOP_K_FEATURES,
        help="Number of selected fingerprint features.",
    )

    # In interactive environments like Colab, argparse can receive unrecognized arguments
    # from the kernel. parse_known_args() allows parsing only the arguments defined
    # in the script and ignores the others.
    args, unknown = parser.parse_known_args()

    results = run_workflow(top_k=args.top_k)
    predictions = results["predictions"]

    print(f"Retrieved PubChem drugs: {len(results['pubchem_df'])}")
    print(f"Fingerprint shape: {results['fingerprints'].shape}")
    print(f"Prediction count: {len(predictions)}")

    print("\nTop predicted compounds:")
    print(
        predictions[
            [
                "compound_name",
                "cid",
                "predicted_probability_active",
                "predicted_label",
            ]
        ].head(10).to_string(index=False)
    )

    print(f"\nLIME files generated: {len(results['lime_files'])}")
    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Fetching PubChem record: prothionamide
Fetching PubChem record: clofazimine
Fetching PubChem record: rifapentine
Fetching PubChem record: pyrazinamide
Fetching PubChem record: atorvastatin
Fetching PubChem record: isoniazid
Fetching PubChem record: pretomanid
Fetching PubChem record: bedaquiline
Fetching PubChem record: doxycycline
Fetching PubChem record: isoniazid sodium methanesulfonate
Fetching PubChem record: ibuprofen
Fetching PubChem record: ethionamide
Fetching PubChem record: rifampicin
Fetching PubChem record: omeprazole
Fetching PubChem record: meropenem
Fetching PubChem record: moxifloxacin
Fetching PubChem record: fluconazole
Fetching PubChem record: azithromycin
Fetching PubChem record: ciprofloxacin
Fetching PubChem record: levofloxacin
Fetching PubChem record: ethambutol
Fetching PubChem record: metformin
Fetching PubChem record: delamanid
Fetching PubChem record: amoxicillin
Fetching PubChem record: rifabutin
Fetching PubChem record: linezolid
Fetching PubChem record: 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but VarianceThreshold was fitted with feature names
  warnings.warn(


Retrieved PubChem drugs: 31
Fingerprint shape: (31, 10146)
Prediction count: 31

Top predicted compounds:
                    compound_name       cid  predicted_probability_active predicted_label
                    prothionamide    666418                      0.638495     active_like
                      doxycycline  54671203                      0.627288     active_like
                       rifampicin 135398735                      0.618995     active_like
                     pyrazinamide      1046                      0.519179     active_like
                      rifapentine 135403821                      0.506389     active_like
                       omeprazole      4594                      0.490939   inactive_like
                      clofazimine      2794                      0.442302   inactive_like
isoniazid sodium methanesulfonate  23691029                      0.431678   inactive_like
                     levofloxacin    149096                      0.406019   inactive